In [1]:
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Model, layers

In [2]:
class TokenAndEmbedding(layers.Layer):
    def __init__(self, max_len: int, vocab_size: int, embed_dim: int):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(0, maxlen, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)

In [3]:
class Encoder(layers.Layer):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, rate: float = 0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim)]
        )
        self.layer_norm1 = layers.Normalization()
        self.layer_norm2 = layers.Normalization()
        self.dropout1 = layers.Dropout(rate=rate)
        self.dropout2 = layers.Dropout(rate=rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layer_norm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layer_norm2(out1 + ffn_output)

In [4]:
class Decoder(layers.Layer):
    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, rate: float = 0.1):
        super().__init__()
        self.att1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.att2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [
                layers.Dense(ff_dim, activation="relu"),
            ]
        )

        self.layer_norm1 = layers.Normalization()
        self.layer_norm2 = layers.Normalization()
        self.layer_norm3 = layers.Normalization()

        self.dropout1 = layers.Dropout(rate=rate)
        self.dropout2 = layers.Dropout(rate=rate)
        self.dropout3 = layers.Dropout(rate=rate)

    def __call__(self, inputs, encoder_outputs, training=False):
        attn1 = self.att1(inputs, inputs, use_causal_mask=False)
        attn1 = self.dropout1(attn1, training=training)
        out1 = self.layer_norm1(attn1 + inputs)

        attn2 = self.att2(query=out1, value=encoder_outputs)
        attn2 = self.dropout2(attn2, training=training)
        out2 = self.layer_norm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        return self.layer_norm3(out2 + ffn_output)

In [5]:
class Seq2Seq(Model):
    def __init__(
        self, num_layers, embed_dim, num_heads, ff_dim, src_vocab, trg_vocab, maxlen
    ):
        super().__init__()
        self.src_emb = TokenAndEmbedding(maxlen, src_vocab, embed_dim)
        self.trg_emb = TokenAndEmbedding(maxlen, trg_vocab, embed_dim)

        self.encoders = [
            Encoder(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ]
        self.decoders = [
            Decoder(embed_dim, num_heads, ff_dim) for _ in range(num_layers)
        ]
        self.final_layer = layers.Dense(trg_vocab)

    def call(self, inputs):
        src, trg = inputs
        enc_output = self.src_emb(src)
        for encoder in self.encoders:
            enc_output = encoder(enc_output)
        dec_output = self.trg_emb(trg)
        for decoder in self.decoders:
            dec_output = decoder(dec_output, encoder_outputs=enc_output)
        return self.final_layer(dec_output)

In [6]:
# --- Hyperparameters ---
SRC_VOCAB = 5000
TGT_VOCAB = 5000
EMBED_DIM = 512
NUM_HEADS = 8
FF_DIM = 512
NUM_LAYERS = 3
MAX_LEN = 100

# Initialize the model
model = Seq2Seq(NUM_LAYERS, EMBED_DIM, NUM_HEADS, FF_DIM, SRC_VOCAB, TGT_VOCAB, MAX_LEN)

# --- Dummy Data ---
# Batch Size: 2, Source Len: 10, Target Len: 12
src_batch = tf.random.uniform((2, 10), minval=0, maxval=SRC_VOCAB, dtype=tf.int32)
tgt_batch = tf.random.uniform((2, 12), minval=0, maxval=TGT_VOCAB, dtype=tf.int32)

# --- Forward Pass ---
logits = model((src_batch, tgt_batch))

print(f"Output shape: {logits.shape}")
# Output: (2, 12, 5000) -> (Batch Size, Target Seq Len, Target Vocab Size)

Output shape: (2, 12, 5000)


In [7]:
import tensorflow as tf

# logits is your (2, 12, 5000) output
predictions = tf.argmax(logits, axis=-1)

print(predictions.shape)
# Output: (2, 12)

print(predictions)
# Output might look like:
# [[  45,  192, 4001,   12, ... (12 words total)],
#  [   8,   34,  911,  844, ... (12 words total)]]

(2, 12)
tf.Tensor(
[[3335  619  615  619 3335 4479  455  935 4609  221    0  221]
 [3529  168 3850 2837 3335 3335 2432 1205 2355 2837 2432 1634]], shape=(2, 12), dtype=int64)
